# Phase 3: Evaluation Pipeline

This notebook measures the quantitative improvement of your LoRA fine-tuned model against the base Llama-3 model using **BLEU, ROUGE, and Cosine Semantic Similarity**.

In [ ]:
!pip install -q -U evaluate sentence-transformers rouge_score

### 1. Load Models and Tokenizer
Upload your zipped `finetuned_model_lora` folder to Colab and unzip it.

In [ ]:
!unzip -q finetuned_model_lora.zip

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
lora_weights_path = "./finetuned_model_lora"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Loading Fine-Tuned LoRA Adapters...")
ft_model = PeftModel.from_pretrained(base_model, lora_weights_path)

### 2. Load Evaluation Metrics

In [ ]:
import evaluate
from sentence_transformers import SentenceTransformer, util
import json

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

### 3. Generate Responses

In [ ]:
def get_risk_classification_prompt(narrative: str) -> str:
    return f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert financial compliance officer. Analyze the following risk narrative and extract the 'Risk Category', a brief 'Risk Summary', and the 'Severity'. Return the output as structured JSON.

### Input:
{narrative}

### Response:
"""

def generate_response(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

with open("financial_risk_dataset.jsonl", "r") as f:
    records = [json.loads(line) for line in f.readlines()[:20]]

base_preds, ft_preds, refs = [], [], []

print("Running inference for evaluation...")
for record in records:
    prompt = get_risk_classification_prompt(record['narrative'])
    
    # Temporarily disable adapters to test base model
    with ft_model.disable_adapter():
        base_preds.append(generate_response(ft_model, tokenizer, prompt))
        
    # Enable adapters for fine-tuned output
    ft_preds.append(generate_response(ft_model, tokenizer, prompt))
    refs.append([record['output']])

### 4. Calculate Improvement

In [ ]:
print("\n--- Base Model Metrics ---")
base_bleu = bleu.compute(predictions=base_preds, references=refs)
base_rouge = rouge.compute(predictions=base_preds, references=refs)
base_emb = similarity_model.encode(base_preds)
ref_emb = similarity_model.encode([r[0] for r in refs])
base_sim = util.cos_sim(base_emb, ref_emb).diag().mean().item()
print(f"BLEU: {base_bleu['bleu']:.4f}, Semantic Sim: {base_sim:.4f}")

print("\n--- Fine-Tuned Model Metrics ---")
ft_bleu = bleu.compute(predictions=ft_preds, references=refs)
ft_rouge = rouge.compute(predictions=ft_preds, references=refs)
ft_emb = similarity_model.encode(ft_preds)
ft_sim = util.cos_sim(ft_emb, ref_emb).diag().mean().item()
print(f"BLEU: {ft_bleu['bleu']:.4f}, Semantic Sim: {ft_sim:.4f}")

improvement = ((ft_sim - base_sim) / base_sim) * 100
print(f"\nTotal Improvement in Semantic Relevance: {improvement:.2f}%")